<a href="https://colab.research.google.com/github/Quratulain786/-R-versus-Python-EDA/blob/main/RM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Data cleaning Machine Learning**
Dataset Description
Overview
The data has been split into two groups:

training set (train.csv)
test set (test.csv)
The training set should be used to build your machine learning models. For the training set, we provide the outcome (also known as the ?ground truth?) for each passenger. Your model will be based on ?features? like passengers? gender and class. You can also use feature engineering to create new features.

The test set should be used to see how well your model performs on unseen data. For the test set, we do not provide the ground truth for each passenger. It is your job to predict these outcomes. For each passenger in the test set, use the model you trained to predict whether or not they survived the sinking of the Titanic.

We also include gender_submission.csv, a set of predictions that assume all and only female passengers survive, as an example of what a submission file should look like.

Data Dictionary
Variable	Definition	Key
survival	Survival	0 = No, 1 = Yes
pclass	Ticket class	1 = 1st, 2 = 2nd, 3 = 3rd
sex	Sex
Age	Age in years
sibsp	# of siblings / spouses aboard the Titanic
parch	# of parents / children aboard the Titanic
ticket	Ticket number
fare	Passenger fare
cabin	Cabin number
embarked	Port of Embarkation	C = Cherbourg, Q = Queenstown, S = Southampton
Variable Notes
pclass: A proxy for socio-economic status (SES)
1st = Upper
2nd = Middle
3rd = Lower
age: Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5
sibsp: The dataset defines family relations in this way...
Sibling = brother, sister, stepbrother, stepsister
Spouse = husband, wife (mistresses and fianc?s were ignored)
parch: The dataset defines family relations in this way...
Parent = mother, father
Child = daughter, son, stepdaughter, stepson
Some children travelled only with a nanny, therefore parch=0 for them.

survival: Survival 0 = No, 1 = Yes
pclass: Ticket class 1 = 1st, 2 = 2nd, 3 = 3rd
sex: Sex
Age: Age in years
sibsp: # of siblings / spouses aboard the Titanic
parch: # of parents / children aboard the Titanic
ticket: Ticket number
fare: Passenger fare
cabin: Cabin number
embarked: Port of Embarkation C = Cherbourg, Q = Queenstown, S = Southampton

In [ ]:
import pandas as pd
import numpy as np
# Load the dataset
df = pd.read_csv('titanic_data.csv')
df.head()
#https://www.geeksforgeeks.org/data-cleansing-introduction/?ref=lbp

In [ ]:
df.describe()

In [ ]:
# Categorical columns
cat_col = [col for col in df.columns if df[col].dtype == 'object']
print('Categorical columns :',cat_col)

In [ ]:
#Numerical columns
num_col = [col for col in df.columns if df[col].dtype != 'object']
print('Numerical columns :',num_col)

In [ ]:
df[cat_col].nunique()

In [ ]:
df['Ticket'].unique()[:50]

In [ ]:
df1 = df.drop(columns=['Name','Ticket'])  # excluding  ticket
print(df1.shape)
print(df.shape)

In [ ]:
# %age of  missing values
round((df1.isnull().sum()/df1.shape[0])*100,2)

In [ ]:
# Creat new dataframe
df2 = df1.drop(columns='Cabin')
df2.dropna(subset=['Embarked'], axis=0, inplace=True)
print(df2.shape)
print(df1.shape)
print(df.shape)

In [ ]:
# Mean imputation  .... Removing Null values
df3 = df2.fillna(df2.Age.mean())
# Let's check the null values again
df3.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt

plt.boxplot(df3['Age'], vert=False)
plt.ylabel('Variable')
plt.xlabel('Age')
plt.title('Box Plot')
plt.show()

In [ ]:
# calculate summary statistics
mean = df3['Age'].mean()
std = df3['Age'].std()

# Calculate the lower and upper bounds
lower_bound = mean - std*2
upper_bound = mean + std*2

print('Lower Bound :',lower_bound)
print('Upper Bound :',upper_bound)

# Drop the outliers
df4 = df3[(df3['Age'] >= lower_bound)
				& (df3['Age'] <= upper_bound)]

In [ ]:
from sklearn.preprocessing import LabelEncoder
cols = ['Sex', 'Embarked']
le = LabelEncoder()

for col in cols:
    df4[col] = le.fit_transform(df4[col])
df4.head()


In [ ]:
X = df4[['Pclass','Sex','Age', 'SibSp','Parch','Fare','Embarked']]
y = df4['Survived']

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# initialising the MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Numerical columns
num_col_ = [col for col in X.columns if X[col].dtype != 'object']
x1 = X
# learning the statistical parameters for each of the data and transforming
x1[num_col_] = scaler.fit_transform(x1[num_col_])
x1.head()

In [ ]:
df4[["Sex", "Survived"]].groupby(['Sex'], as_index=False).mean().sort_values(by='Survived', ascending=False)

In [ ]:
df4['Sex'].value_counts()

In [ ]:
df4[["Age", "Survived"]].groupby(['Age'], as_index=False).mean().sort_values(by='Survived', ascending=False)

In [ ]:
#after data engineering
pd.crosstab([df4.Embarked,df4.Pclass],[df4.Sex,df4.Survived],margins=True).style.background_gradient(cmap='summer_r')

In [ ]:
#Without data engineering
pd.crosstab([df.Embarked,df.Pclass],[df.Sex,df.Survived],margins=True).style.background_gradient(cmap='summer_r')

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.8)

In [ ]:
dfs = [X_train, X_test]

In [ ]:
# Now Data have no missing values
# function to analyze each column of the dataframe
def display_missing(df4):
    for col in df4.columns.tolist():
        print('{} column missing values: {}'.format(col, df4[col].isnull().sum()))
    print('\n')

for df in dfs:
    print('{}'.format(df4))
    display_missing(df4)

In [ ]:
!pip install seaborn matplotlib
import seaborn as sns  # statistical data visualization
sns.heatmap(X_train.corr(), annot=True, cmap='RdYlGn', linewidths=0.2)
fig = plt.gcf()
fig.set_size_inches(10, 8)
plt.show()

# **Support Vector Machine**

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Initialize and train the Decision Tree model
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

# Calculate the training and testing accuracy
train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print("Training Accuracy: ", train_accuracy)
print("Testing Accuracy: ", test_accuracy)

In [ ]:
# Predict the labels for the test set

y_pred = model.predict(X_test)

# Print the predictions
print("Predictions on the test set:", y_pred)

In [ ]:
# Create the confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
conf_matrix = confusion_matrix(y_test, y_pred)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

# Tuining the best parameter

In [ ]:
#Here the model is not well-fitted based on train and test accuracy. so we have to tune best parameters to solve the problem.
# Define the parameter grid
from sklearn.model_selection import GridSearchCV
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 50, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the model
model = DecisionTreeClassifier()

# Initialize GridSearchCV
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')

# Fit the GridSearchCV to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters and estimator
best_params = grid_search.best_params_
best_estimator = grid_search.best_estimator_

print('Best Hyperparameters:', best_params)


In [ ]:
# Initialize the model with specified hyperparameters
model = DecisionTreeClassifier(criterion='entropy', max_depth=6, min_samples_leaf=8, min_samples_split=20)

# Fit the model to the training data
model.fit(X_train, y_train)

# Calculate and print the training and testing accuracy
print("Training Accuracy:", model.score(X_train, y_train))
print("Testing Accuracy:", model.score(X_test, y_test))

In [ ]:
# Predict the labels for the test set
y
y_pred = model.predict(X_test)

# Print the predictions
print("Predictions on the test set:", y_pred)

In [ ]:
# Create the confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=model.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()